In [56]:
""" import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
"""

' import torch\nprint(torch.cuda.is_available(), torch.cuda.get_device_name(0))\n'

# Create and run a local RAG pipeline

## Goals for RAG

* Retrieval - Find relevant info given a query, e.g. "what are the macronutrients and what do they do?" -> retrieves passages of text realted to the macronutrients from a nutrition textbok.
* Augmented - Take the relevant info and augment our input(prompt) to an LLM with that relevant info
* Generation - Take the first two steps and pass them to an LLM for generative outputs

## Main Goal
improve the outputs of LLMs by:
1. Preventing hallucinations
2. Work with custom data


## Use Cases:
* Textbook Q&A - find releveant passages to the questions you have.

In [57]:
import os
import requests

# Get PDF document path
pdf_path = "sustainability-action-plan.pdf"

# Download PDF
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    #Enter the URL of the PDF
    url = "https://cityofwinterpark.org/docs/departments/natural-resources-sustainability/sustainability-division/sustainability-action-plan/sustainability-action-plan-annual-report-2023.pdf"

    #The local filename to save the downloaded file
    filename=pdf_path

    # Browser-like User-Agent required: the site returns 403 for python-requests' default
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


    #Send a GET request to the URL
    response = requests.get(url, headers=headers, timeout=30)

    #Check if the request was successful
    if response.status_code ==200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)

        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")

else:
    print(f"File {pdf_path} exists.")



File sustainability-action-plan.pdf exists.


In [58]:
%pip install tqdm
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [59]:
import pymupdf # requires: !pip install PyMuPDF
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text= text.replace("\n"," ").strip()

    return cleaned_text

PRINTED_PAGE_OFFSET = 2
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path)
    pages_and_texts=[]
    
    for page_number, page in tqdm(enumerate(doc)):
        if page_number < PRINTED_PAGE_OFFSET:
            continue # skip the two pages before the title page
        
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - PRINTED_PAGE_OFFSET + 1,
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count": len(text.split(". ")),
                                "page_token_count": len(text) / 4, # 1 token = ~ 4 characters
                                "text": text})

    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': 1,
  'page_char_count': 406,
  'page_word_count': 114,
  'page_sentence_count': 1,
  'page_token_count': 101.5,
  'text': 'City of Winter Park  Sustainability Action Plan                  January 24th, 2024                            Presented by:   Gloria Eby, Natural Resources and Sustainability Director     Sara Miller, Sustainability Manager   Mia Brady, Sustainability Specialist   Keep Winter Park Beautiful & Sustainable Advisory Board           Certified WCAG 2.1 Level AA compliant and usable with JAWS and NVDA by SWH'},
 {'page_number': 2,
  'page_char_count': 2873,
  'page_word_count': 155,
  'page_sentence_count': 22,
  'page_token_count': 718.25,
  'text': '2    Contents   Background ........................................................................................................................................... 3  Vision and Purpose ......................................................................................................................

In [60]:
import pandas as pd
pages_and_texts[9]

{'page_number': 10,
 'page_char_count': 1771,
 'page_word_count': 277,
 'page_sentence_count': 4,
 'page_token_count': 442.75,
 'text': '10  o Parks and Recreation summer camp program incorporates Orange County Public Schools (OCPS) Summer Food Program o Summer Feeding - Orange County Public Schools (ocps.net) • Winter Park Library o Master gardening basics:  5 classes, partnership with UF/IFAS Extension orange County o Seed library: 3 grow seasons; 9,000 seeds distributed per year • Rollins College o Food pantry for students – Students can either come to the pantry and take what they need or place an order online, which will be processed at the C-Store and stored in a pre-packed bag o Sam Justice is staff contact - sjustice@rollins.edu o Urban farm collaboration - providing fresh produce to the food pantry on a regular basis • SeniorsFirst o https://www.seniorsfirstinc.org/ o Provides meals-on-wheels to seniors in Winter Park • Winter Park High School o Willies Mart – “Willie’s Mart p

In [61]:

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,text
0,1,406,114,1,101.50,City of Winter Park Sustainability Action Pla...
1,2,2873,155,22,718.25,2 Contents Background ......................
2,3,2448,420,12,612.00,3 Background Vision and Purpose The 2022 Su...
3,4,2954,505,16,738.50,"4 In 2012, the city’s Environmental Review an..."
4,5,2551,411,7,637.75,5 indicators to benchmark city status in effo...


## Table Preprocessing

The goal is to extract tables from the PDF while preserving the relationships between column headers and row values

In [62]:
doc = pymupdf.open(pdf_path)

for page_number, page in enumerate(doc):
    tables = page.find_tables().tables

    if tables:
        print(f"Page {page_number}: {len(tables)} table(s) found")

Page 5: 1 table(s) found
Page 6: 1 table(s) found
Page 12: 1 table(s) found
Page 13: 1 table(s) found
Page 14: 1 table(s) found
Page 15: 1 table(s) found
Page 18: 2 table(s) found
Page 19: 1 table(s) found
Page 20: 1 table(s) found
Page 22: 2 table(s) found
Page 23: 1 table(s) found
Page 25: 2 table(s) found
Page 26: 1 table(s) found
Page 27: 1 table(s) found
Page 28: 1 table(s) found
Page 31: 1 table(s) found
Page 32: 1 table(s) found
Page 33: 1 table(s) found
Page 37: 2 table(s) found
Page 38: 1 table(s) found
Page 39: 1 table(s) found
Page 40: 1 table(s) found
Page 42: 2 table(s) found
Page 43: 1 table(s) found
Page 45: 2 table(s) found
Page 46: 1 table(s) found
Page 47: 1 table(s) found
Page 51: 1 table(s) found
Page 52: 2 table(s) found
Page 55: 2 table(s) found
Page 57: 1 table(s) found


In [63]:
# Automatically extract every table

all_tables = []

for page_number, page in enumerate(doc):
    tables = page.find_tables(strategy="lines_strict").tables

    for table_index, table in enumerate(tables):
        table_df = table.to_pandas()

        all_tables.append({
            "page_number" : page_number,
            "table_index": table_index,
            "data": table_df
        })

print(f"Extracted {len(all_tables)} tables")

Extracted 39 tables


# Inspect pages with Tables
## Goals:
1. Turn the tables into dataframe to inspect the actual rows and columns instead of treating the table as one long string

In [64]:
# Inspect the first table
display(all_tables[0]["data"])
# Check which page it came from
print(all_tables[0]["page_number"])

,2012-2013 KWPB&S Board Members,Col1,2014-2015 KWPB&S Board Members,Col3
0,"Mary Dipboye, Chair",James (Bob) Robinson,"Michael Poole, Chair",Mark Roush
1,"Stephen Pategas, V. Chair",Pat Schoknecht,"Stephen Pategas, V. Chair",Bruce Thomas
2,Michele Hipp,Julia Tensfeldt,Michele Hipp,Steven DiClemente
3,Michael Poole,Kent Tse,Raymond Randall,Mary Dipboye
4,Raymond Randall,Laura Walda,Pat Schoknecht,John Tapp
5,John Rife,Carol Kostick,Julia Tensfeldt,Fred Kosiewski
6,Lucy Roberts,Mark Roush,Laura Walda,Cathy Blanton
7,Joseph Robillard,,Carol Shenck (Kostick),


5


In [80]:
df = all_tables[0]["data"].copy()

# Combine each pair of columns vertically and remove empty cells
board_2012 = pd.concat([df.iloc[:, 0], df.iloc[:, 1]]).astype(str).str.strip()
board_2012 = board_2012[board_2012.ne("") & board_2012.ne("nan")].reset_index(drop=True)

board_2014 = pd.concat([df.iloc[:, 2], df.iloc[:, 3]]).astype(str).str.strip()
board_2014 = board_2014[board_2014.ne("") & board_2014.ne("nan")].reset_index(drop=True)

# Create the cleaned table
cleaned_df = pd.DataFrame({
    "2012-2013 KWPB&S Board Members": board_2012,
    "2014-2015 KWPB&S Board Members": board_2014
})

# Save it back into all_tables
all_tables[0]["data"] = cleaned_df

display(all_tables[0]["data"])

,2012-2013 KWPB&S Board Members,2014-2015 KWPB&S Board Members
0,"Mary Dipboye, Chair","Michael Poole, Chair"
1,"Stephen Pategas, V. Chair","Stephen Pategas, V. Chair"
2,Michele Hipp,Michele Hipp
3,Michael Poole,Raymond Randall
4,Raymond Randall,Pat Schoknecht
5,John Rife,Julia Tensfeldt
6,Lucy Roberts,Laura Walda
7,Joseph Robillard,Carol Shenck (Kostick)
8,James (Bob) Robinson,Mark Roush
9,Pat Schoknecht,Bruce Thomas


In [65]:
display(all_tables[3]["data"])
print(all_tables[3]["page_number"])

,Projected\nImplementation\nYear,Action,Accomplished,Status,Responsible\nDepartment/\nDivision
0,2022,Conduct Renewable Energy\nFeasibility Study,Yes,Completed,"Sustainability,\nQuanta, & WP\nElectric Utilit..."
1,2023,Establish community grants\nfor food security ...,No,None,Sustainability
2,2022,Implement an Artificial Turf\nOrdinance which ...,Yes,Yes,"Sustainability,\nPublic Works"
3,2022,Adopt Backyard Chicken\nProgram (exp. Septembe...,Yes,Yes,"Sustainability,\nPlanning"
4,2023,"Upon feasibility study\nresults, pass resoluti...",No,Initiated,"Sustainability,\nWP Electric\nUtility"
5,2023,Conduct a Climate Risk and\nVulnerability Asse...,No,Initiated,"Natural\nResources &\nSustainability,\nEmergen..."


13


In [66]:
display(all_tables[13]["data"])
print(all_tables[13]["page_number"])

,Col0,Indicator Description,2012\nBaseline,2021\nStatus,2022\nStatus,2025\nTarget,2035\nTarget
0,W-1,Water usage in residential buildings\n[gallons...,"127,777","130,849","127,606",TBD,TBD
1,W-2,Water usage in commercial buildings\n[gallons/...,"294,098","303,788","315,099",TBD,TBD
2,W-3,Reclaimed water usage [million\ngallons/year]2,144.5,91.61,87.97,10%\nincrease,30%\nincrease
3,W-4,Residential Rebates provided\nannually,-,34,19,75,150


25


In [67]:
display(all_tables[32]["data"])
print(all_tables[32]["page_number"])

,Climate Resiliency,Conduct Renewable Energy Feasibility Study
0,Climate Resiliency,Implement an Artificial Turf Ordinance which w...
1,Climate Resiliency,Adopt Backyard Chicken Program (exp. September...
2,Water,Create water conservation education to residen...
3,Water,Increase public awareness of Florida-friendly ...
4,Community Engagement and Green\nEconomy,Re-participate in America In Bloom’s annual na...
5,Local Government Operations,Solicit proposals for energy conservation audi...
6,Local Government Operations,Conduct Renewable Energy Feasibility Study – 5...
7,Natural Resources,Provide education on pollution prevention of n...
8,Natural Resources,Establish Lakes Newsletter to inform citizens ...
9,Waste Management,Provide composting education and backyard comp...


47


In [68]:
display(all_tables[34]["data"])
print(all_tables[34]["page_number"])

,Battery Energy Storage Technology,2023,2035,2042,2050
0,Residential BESS–2.5 hr,"$1,443","$1,359","$1,359","$1,359"
1,Commercial BESS–4 hr,$774,$729,$729,$729
2,Utility Scale BESS–4 hr (ESS-4),$514,$484,$484,$484


52


In [69]:
display(all_tables[35]["data"])
print(all_tables[35]["page_number"])

,Solar PV Technology,2023,2035,2042,2050
0,Residential Rooftop Solar,"$3,400","$2,892","$2,892","$2,892"
1,"Commercial Solar PV, Rooftop (Dsolar-CommRoof)","$2,121","$1,804","$1,804","$1,804"
2,"Commercial Solar PV, Ground-mount (Dsolar-Comm...","$2,236","$1,902","$1,902","$1,902"
3,Utility Scale Solar PV (USolar),"$1,331","$1,253","$1,253","$1,253"
4,Green Hydrogen-Fueled CT (CT-Hydrogen),"$1,561","$1,828","$1,828","$1,828"
5,Biofuel Internal Combustion Reciprocating Engi...,$520,$552,$552,$552


52


In [70]:
display(all_tables[36]["data"])
print(all_tables[36]["page_number"])

,Year,Scenario\n3A,Scenario\n4A,Scenario\n4B
0,20-year PVRR – 2023 to 2042 ($M),505,637,558
1,LCOE based on 20-year PVRR ($/MWh),90,113,99


55


In [71]:
display(all_tables[37]["data"])
print(all_tables[37]["page_number"])

,Year,Cumulative\nCity Rooftop\nPV (MW),Cumulative\nCity Ground\nMount PV\n(MW),Cumulative\nTotal MW),Cumulative\nCost ($M)
0,2035,0.98,0.98,1.96,$3.8
1,2042,0.98,2.95,3.94,$7.5\n$7.5
2,2050,0.98,2.95,3.94,None


55


In [72]:
display(all_tables[38]["data"])
print(all_tables[38]["page_number"])

,Year,Annualized\nCost\n($/MWh),Annualized\nCost\n($/MWh)\nescalated at\n3% after\n2042,LCOE\n($/MWh)\nwith a 3%\nescalation,Maximum\nAcceptable\nAnnualized\nPower Costs
0,2023,$72,,$70,
1,2024,$72,,$73,
2,2025,$72,,$75,
3,2026,$74,,$77,
4,2027,$75,,$79,
5,2028,$77,,$82,
6,2029,$78,,$84,
7,2030,$77,,$87,
8,2031,$76,,$89,
9,2032,$80,,$92,


57


In [74]:
import random
random.sample(pages_and_texts, k=3)

[{'page_number': 17,
  'page_char_count': 1748,
  'page_word_count': 354,
  'page_sentence_count': 2,
  'page_token_count': 437.0,
  'text': '17  INDICATORS  Indicator Description  2023  Baseline  2024  Target  2026  Target  2030  Target  2035  Target  ES-1  Annual renewable energy  contribution to the total  electric energy supply (%)  21.72%  21.72%  21.72%  50%  80%  ES-2  City residential rates as  compared to FL public utilities  (city rates/average of  comparable utilities, measured in %)    <100%  <105%  <105%  <105%  <105%  ES-3  EE and DR program milestones  Not  applicable  Complete studies  and develop  implementation  plan for customer  EE and DR programs  Implement  programs  for  customer  EE and DR  ES-4  Milestones to implement  changes to rate to enable  substantial growth in customer  owned distributed solar,  distributed storage and electric  vehicle charging  Not  applicable  Complete rate  studies and develop  rate design options  to enable growth in  distributed s

In [75]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count
count,57.0,57.00,57.00,57.00,57.00
mean,29.0,2050.35,353.84,9.28,512.59
std,16.6,701.22,113.45,7.93,175.30
min,1.0,406.00,114.00,1.00,101.50
25%,15.0,1559.00,279.00,2.00,389.75
50%,29.0,1958.00,366.00,7.00,489.50
75%,43.0,2421.00,420.00,15.00,605.25
max,57.0,3876.00,639.00,33.00,969.00


# Further text processing (splitting pages into sentences)

In [ ]:
import 

In [76]:
# Drop Heading and Table of contents
records_df = records_df[~records_df.page_number.isin([1, 2])].reset_index(drop=True)

NameError: name 'records_df' is not defined

Interpreting the token count
Caution:

1.  LLMs don't deal with infinite tokens
2. Embedding models don't deal with infinite tokens